In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 130

In [46]:
data = 'https://raw.githubusercontent.com/gastonstat/CreditScoring/master/CreditScoring.csv'
!wget $data

--2026-05-29 12:56:12--  https://raw.githubusercontent.com/gastonstat/CreditScoring/master/CreditScoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 182489 (178K) [text/plain]
Saving to: ‘CreditScoring.csv.8’

CreditScoring.csv.8 100%[===================>] 178.21K  --.-KB/s    in 0.02s   

2026-05-29 12:56:13 (7.20 MB/s) - ‘CreditScoring.csv.8’ saved [182489/182489]



In [47]:
df = pd.read_csv(data)

In [48]:
df.head()

,Status,Seniority,Home,Time,Age,Marital,Records,Job,Expenses,Income,Assets,Debt,Amount,Price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


# Data Cleaning and Preparation
- Decoding feature values
- correcting dtypes
- handling null values


In [49]:
df.columns = df.columns.str.lower()
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


In [50]:
df['status'].unique()

array([1, 2, 0])

In [51]:
df.dtypes

,0
status,int64
seniority,int64
home,int64
time,int64
age,int64
marital,int64
records,int64
job,int64
expenses,int64
income,int64


In [52]:
df['status'] = df['status'].astype(int).map({1: 'ok', 2: 'default', 0: 'unk'})



In [62]:
df['marital'].unique()


array([2, 3, 1, 4, 5, 0])

In [63]:
df['marital'] = df['marital'].map({ 1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'})

In [65]:
df['home'] = df['home'].map({ 1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'})

df['records'] = df['records'].map({ 1: 'no',
    2: 'yes',
    0: 'unk'})
df['job'] = df['job'].map({
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
})

In [67]:
df[['status', 'seniority', 'home', 'marital', 'job']] = df[['status', 'seniority', 'home', 'marital', 'job']].astype('str')

In [69]:

num_cols = list(df.dtypes[df.dtypes == 'int'].index)
num_cols


['time', 'age', 'expenses', 'income', 'assets', 'debt', 'amount', 'price']

In [70]:
#decoding null values in numerical columns
for col in num_cols:
  df[col] = df[col].replace(df[col].max(), np.nan)


In [53]:
# Data Validation Framework
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

len(df_train), len(df_test), len(df_val)

(2673, 891, 891)

In [54]:
y_train = df_train['status'].values
y_test = df_test['status'].values
y_val = df_val['status'].values